In [ ]:
# Overfitting - producing a model that performs well on the data you train it on but generalizes poorly to any new data.
# Underfitting - producing a model that doesn't perform well even on tarining data.

# Models that are too complex lead to overfitting, to avoid
# Split dataset into training and testing e.g 60% training, 40% testing performance

In [ ]:
import random
from typing import TypeVar, List, Tuple

X = TypeVar('X') # generic type to represent a data point

def split_data(data: List[X], prob: float) -> Tuple[List[X], List[X]]:
    """Split data into fractions [prob, 1 - prob]"""
    data = data[:]                  # make a shallow copy
    random.shuffle(data)            # because shuffle modifies the list.
    cut = int(len(data) * prob)     # Use prob to find cut-off
    return data[:cut], data[cut:]   # and split the shuffled list there.


In [3]:
data = [n for n in range(1000)]
train, test = split_data(data, 0.75)

assert len(train) == 750
assert len(test) == 250

assert sorted(test + train) == data

In [ ]:
# We'll pair input variables and output variables
# In that case we need to make sure to put correponding values together in either the training data ot testing data

Y = TypeVar('Y') # generic type to represent output varibales

def train_test_split(
        xs: List[X], 
        ys: List[Y], 
        test_pct: float
    ) -> Tuple[List[X], List[X], List[Y], List[Y]]:
    """Split paired input and output data into training and testing sets.

    Args:
        xs: Input data.
        ys: Corresponding output data.
        test_pct: Fraction of data reserved for testing.

    Returns:
        Training inputs, testing inputs, training outputs, and testing outputs.
    """
    idxs = [i for i in range(len(xs))]
    train_idxs, test_idxs = split_data(idxs, 1 - test_pct)

    return (
        [xs[i] for i in train_idxs],    # x_train
        [xs[i] for i in test_idxs],     # x_test
        [ys[i] for i in train_idxs],    # y_train
        [ys[i] for i in test_idxs]      # y_test
    )



In [6]:
xs = [x for x in range(1000)]   # xs are 1 ... 1000
ys = [2 * x for x in xs]        # each y_i is twice x_i
x_train, x_test, y_train, y_test = train_test_split(xs, ys, 0.25)

# Check that the proportions are correct
assert len(x_train) == len(y_train) == 750
assert len(x_test) == len(y_test) == 250

# Check that the corresponding data points are paired correctly
assert all(y == 2 * x for x, y in zip(x_train, y_train))
assert all(y == 2 * x for x, y in zip(x_test, y_test))

In [ ]:
# After which we can do 

class SomekindOfModel:
    """A basic model with training and testing operations."""
    
    def train(self, xs: List[X], ys: List[X]):
        pass

    def test(self, xs: List[Y], ys: List[Y]):
        pass

model = SomekindOfModel()
x_train, x_test, y_train, y_test = train_test_split(xs, ys, 0.33)
model.train(x_train, y_train)
performance = model.test(x_test, y_test)

In [ ]:
# If model was overfit to the training data, then it will hopefully perform poorly on the (completely separate) test data.

In [ ]:
# CORRECTNESS

# - We dont typically use "accuracy" to measure how good a (binary classification) model is.
# - Models to make a binary judgement, Is thus email spam? Should we hire candidate? is this air traveller secretly a terrorist?

# - Every datapoint lies in one of four categories:
# - True positive > "This email is spam, and we correctly predicted spam."
# - False Positive (T1E) > "This email is not spam, but we predicted spam."
# - False Negative (T2E) > "This email is spam, but we predicted not spam."
# - True negative > "This email is not spam, we crrectly predicted not spam."

In [ ]:
# Predict leukemia if and only if the baby is named Luke

# - 5 out of 1000 babies named Luke > 0.5%
# - 14 out of 1000 lifetime leukimia prevelance > 1.4%

# - If we believe these 2 factors are independent and apply "Luke is for Leukimia" test to 1 million people, we'd expect to get a confusion matrix

#              Leukemia     |   No Leukimia |   Total
# ```````````````````````````````````````````````````
# "Luke"       70           |   4,930       |  5,000 
#````````````````````````````````````````````````````
# "Not Luke"   13,930       | 981,070       | 995,000
#````````````````````````````````````````````````````
# "Total"      14000        | 986,000       | 1,000,000
#``````````````````````````````````````````````````````

In [11]:
# use the data to compute various statistics about model perfomance

## Accuracy - is the fraction of correct predictions

def accuracy(tp: int, fp: int, fn: int, tn: int) -> float:
    correct = tp + tn
    total = tp + fp + fn + tn
    return correct/total

assert accuracy(70, 4930, 13930, 981070) == 0.98114

## Precision - measures how accurate our positive predictions were

def precision(tp: int, fp: int, fn: int, tn: int) -> float:
    return tp / (tp + fp)

assert precision(70, 4930, 13930, 981070) == 0.014

## Recall - measures what fraction of the positives our model identified

def recall(tp: int, fp: int, fn: int, tn: int) -> float:
    return tp / (tp + fn)

assert recall(70, 4930, 13930, 981070) == 0.005

## Sometimes precision and recall are combined into the F1 score, which is the harmonic mean of precision and recall and necessarily lies between them

def f1_score(tp: int, fp: int, fn: int, tn: int) -> float:
    p = precision(tp, fp, fn, tn)
    r = recall(tp, fp, fn, tn)

    return 2 * p * r / (p + r)

print(f"F1-Score: {f1_score(70, 4930, 13930, 981070)}")



F1-Score: 0.00736842105263158


In [ ]:
# - Usually, choice of a model involves a tradeoff between recall and precision.
# - A model that predicts "YES" when its even a lil bit confident will probably have a high recall but a low precision;
# - a model that predicts "YES" when its extremely confident is likely to have a low recall and a high precision

# - Alternatively, think of this as a tradeoff between fasle positives and false negatives
# - Saying 'YES' too often will give you lots of false positives; saying 'NO' too often will give lots of false negatives.



In [ ]:
# THE BIAS-VARIANCE TRADEOFF

# - Another way of thinking of overfitting problem is  tradeoff between bias and variance
# - Both are measures of what would happen if you were to retrain your model many times on different sets of training data (from the same larger population)

# HIGH BIAS and LOW VARIANCE typically correspond to underfitting
# LOW BIAS and HIGH VARIANCE correspond to overfitting

# - If model has high bias, one thing, try adding more features.
# - If model has high variance, (1) you can remove features, (2) obtain more data

# - Holding model complexity constant, the more data you have, the harder is it to overfit. 
# - However, more data wont help with bias. 
# - If model doesn't use enough features to capture regularities in the data, throwing more data at it won't help. 

In [ ]:
# FEATURE EXTRACTION AND SELECTION

# - Data with not enough features, model likely to underfit
# - Data with too many features, it's easy to overfit

# Features are whatever inputs we provide to our model.
# e.g. If were to predict someone's salary based on years of expreience, then years of experience is the only feature we have.
# 
# - Imagine builiding a spam filter to predict whether an email is junk or not. Most models wont know what to do with a raw email, which is just a collection of text. We'll have to extract features. E.g.
# - Does the email contain the word Viagra? (Yes or No)
# - How many times does the letter d appear? (N)
# - What was the domain of the sender?  (choice from a discrete set of options)